# FASTSim 2 to FASTSim 3 Migration Guide

FASTSim 3 is the newest version of FASTSim. This guide is for existing FASTSim 2 users migrating to FASTSim 3.

You will learn how to:

- Install and import FASTSim 3
- Load vehicles and drive cycles (including converting your existing FASTSim 2 vehicle files)
- Run simulations
- Read scalar parameters and time-series results
- Modify vehicle parameters
- Identify name and path changes for FASTSim vehicle fields, with provided FASTSim 2 to FASTSim 3 reference guide mapping FASTSim 2 fields to their FASTSim 3 counterparts

## What Changed?

The biggest change is how vehicle data is stored:

- **FASTSim 2** used a flat structure - every parameter (`fc_max_kw`, `ess_max_kwh`, `mc_max_kw`, …) was a top-level attribute on the `Vehicle` object, and every powertrain field existed on every vehicle regardless of whether it applied (e.g. conventional vehicles still had a battery field).
- **FASTSim 3** uses a **nested hierarchical structure** where each vehicle only contains the fields relevant to its powertrain (`Conv`, `HEV`, `PHEV`, or `BEV`). Combining incompatible fields (e.g. a conventional vehicle with a battery) is now impossible. This allows for safer, clearer data handling.

Other important differences:

- **SI units are baked into field names** (`_watts`, `_joules`, `_kilograms`, `_meters`, `_seconds`). This means some variable units have been updated (no more kW or mph units), and all variables now have clear and uniformly labeled units.
- **Simulation results are accessed via `to_dataframe()` and `to_pydict()`** rather than as direct array attributes on the `SimDrive` object.
- **Method to modify vehicle attributes has changed.** Modifying a field now requires first changing to a pydict, modifying, then changing back.
- **Configurable save intervals** (`set_save_interval`) - disabling per-step recording gives roughly a 10× speedup and decreased memory usage.
- **Thermal modeling** (cabin, HVAC, battery, engine) is now supported.

## Quick Reference

The table below covers common FASTSim actions.

| Task | FASTSim 2 | FASTSim 3 |
| --- | --- | --- |
| Import | `import fastsim as fsim` | `import fastsim as fsim` |
| Load vehicle from resource | `fsim.Vehicle.from_resource("2012_Ford_Fusion.yaml")` | `fsim.Vehicle.from_resource("2012_Ford_Fusion.yaml")` |
| Load vehicle from database | `fsim.vehicle.Vehicle.from_vehdb` | `fsim.Vehicle.from_db` (see [](user-guide/vehicle-models/loading-vehicles.ipynb) for more information) |
| Vehicle from file | `fsim.vehicle.Vehicle.from_file("veh.yaml")` | `fsim.Vehicle.from_file("veh.yaml")` |
| Load a FASTSim 2 file into FASTSim 3 | N/A | `fsim.Vehicle.from_file("veh_f2.yaml")` |
| List vehicles in resources | `fsim.Vehicle.list_resources()` | `fsim.Vehicle.list_resources()` |
| Load cycle from resource | `fsim.cycle.Cycle.from_file("udds")` | `fsim.Cycle.from_resource("udds.csv")`|
| Load cycle from file | `fsim.cycle.Cycle.from_file("cycle.csv")` | `fsim.Cycle.from_file("cycle.csv")` |
| Access Powertrain type | `veh.veh_pt_type` (string) | `veh.veh_type()` → `"Conv"` / `"HEV"` / `"PHEV"` / `"BEV"` |
| Read a vehicle variable value | `veh.fc_max_kw` | `veh.to_pydict(flatten=True)["pt_type.Conv.fc.pwr_out_max_watts"]` |
| Modify a parameter | `veh.fc_max_kw = 100` | `d = veh.to_pydict(flatten=False)`<br>`d["pt_type"]["Conv"]["fc"]["pwr_out_max_watts"] = 100`<br>`veh = fsim.Vehicle.from_pydict(d)` |
| Create simulation | `fsim.simdrive.SimDrive(cyc, veh)` | `fsim.SimDrive(veh, cyc)` (**argument order reversed**) |
| Run simulation | `sd.sim_drive()` | `sd.walk()` |
| Access time-series result | `sd.fc_kw_in_ach` (array attribute) | `sd.to_dataframe()["veh.pt_type.Conv.fc.history.pwr_fuel_watts"]` |
| Access scalar cumulative result | `sd.fs_kwh_out_ach[-1]` | `sd.to_pydict(flatten=True)["veh.pt_type.Conv.fc.state.energy_fuel_joules"]` |
| Configure save interval | (save interval always 1) | `veh.set_save_interval(1)` / `veh.set_save_interval(None)` |
| Save vehicle | `veh.to_file("veh.yaml")` | `veh.to_file("veh.yaml")` |

## Installation and Imports

FASTSim 3 is installed the same way as FASTSim 2:

```bash
pip install fastsim
```

The top-level import is unchanged, but many classes have moved to the `fastsim` root namespace (they used to live in submodules like `fastsim.vehicle` and `fastsim.simdrive`):

In [ ]:
import fastsim as fsim

# FASTSim 2: fsim.vehicle.Vehicle, fsim.cycle.Cycle, fsim.simdrive.SimDrive
# FASTSim 3: fsim.Vehicle,        fsim.Cycle,        fsim.SimDrive

# Accessing FASTSim 3 classes
print("Vehicle :", fsim.Vehicle)
print("Cycle   :", fsim.Cycle)
print("SimDrive:", fsim.SimDrive)

Vehicle : <class 'fastsim.Vehicle'>
Cycle   : <class 'fastsim.Cycle'>
SimDrive: <class 'fastsim.SimDrive'>


## Loading Vehicles

FASTSim 2 used YAML resource files, custom CSV files, or integer database IDs (`from_vehdb(10)`). FASTSim 3 uses named YAML resources, customized YAML files, or a YAML vehicle database accessible through `from_vehdb()` (see more in [](user-guide/vehicle-models/loading-vehicles.ipynb)).

In [ ]:
# List all vehicles in FASTSim 3 resources
for name in fsim.Vehicle.list_resources():
    print(name)

In [ ]:
# Load a vehicle from FASTSim 3 resources
veh = fsim.Vehicle.from_resource("2012_Ford_Fusion.yaml")

# You can also load from a YAML file on disk:
# veh = fsim.Vehicle.from_file("my_vehicle.yaml")

### Converting a FASTSim 2 vehicle file

If you have existing FASTSim 2 vehicle YAMLs, use `Vehicle.from_file` to load them directly as FASTSim 3 vehicles. This is the same function to load regular FASTSim 3 vehicle YAML files, and will automatically detect and read in both FASTSim 2 and FASTSim 3 vehicle files. You can then re-serialize them as FASTSim 3 YAMLs (via `veh.to_file(...)`) for future use.

Note that not every FASTSim 2 field has a 1-to-1 match in FASTSim 3. See the [Field-Mapping Reference](#field-mapping-reference-guide) tables at the end of this guide for details on how FASTSim 2 vehicle parameters line up with FASTSim 3 parameters.

How to read a FASTSim 2 vehicle into FASTSim 3:
```python
veh_from_f2 = fsim.Vehicle.from_file("fastsim_2_vehicle.yaml")
```

Then, save it to a FASTSim 3 vehicle YAML file for future use:
```python
veh_from_f2.to_file("converted_vehicle.yaml")
```

## Reading Vehicle Parameters

In FASTSim 2 you accessed parameters as flat attributes on the `Vehicle` object:

```python
# FASTSim 2
veh.fc_max_kw          # 130.5
veh.ess_max_kwh        # not applicable - 0.0 for conventional
veh.drag_coef          # 0.393
veh.veh_kg             # 1644
```

In FASTSim 3 the recommended approach is to call `to_pydict(flatten=True)`, which returns a flat `dict` whose keys use dot-separated paths matching the nested vehicle structure. See the [Field-Mapping Reference](#field-mapping-reference-guide) section of this guide for a list of FASTSim 2 vehicle variables and their corresponding FASTSim 3 variable paths/names, since in many cases both paths to variables and names have been updated.

In [ ]:
d = veh.to_pydict(flatten=True)

print("name             :", d["name"])
print("year             :", d["year"])
print("mass_kilograms   :", d["mass_kilograms"], "kg")
print("drag_coef        :", d["chassis.drag_coef"])
print("frontal area     :", d["chassis.frontal_area_square_meters"], "m^2")
print("FC peak power    :", d["pt_type.Conv.fc.pwr_out_max_watts"] / 1e3, "kW")
print("FS energy capac. :", d["pt_type.Conv.fs.energy_capacity_joules"] / 3.6e6, "kWh")
print("aux base load    :", d["pwr_aux_base_watts"], "W")

For quick inspection, FASTSim 3 exposes convenience attributes for the main powertrain components:

- `veh.fc` - fuel converter (Conv, HEV, PHEV)
- `veh.res` - reversible energy storage / battery (HEV, PHEV, BEV)
- `veh.em` - electric machine / motor (HEV, PHEV, BEV)

In [ ]:
print(veh.fc)

### Unit updates

FASTSim 3 field names always include the unit as a suffix, so there is no ambiguity. When translating FASTSim 2 code, remember to convert:

| Quantity | FASTSim 2 unit | FASTSim 3 unit |
| --- | --- | --- |
| Power | kW | W |
| Energy | kWh | J |
| Speed | mph | m/s |
| Mass, length, time, temp | kg, m, s, K | same |

NOTE: for FASTSim 2 vehicles converted into FASTSim 3 vehicles using `Vehicle.from_file`, these updates happen automatically.

## Modifying Vehicle Parameters

**In FASTSim 2**, you could assign values to variables directly:

```python
# FASTSim 2
veh.fc_max_kw = 150
veh.drag_coef = 0.30
```

**In FASTSim 3**, `Vehicle` objects are immutable from Python. To modify a field, convert to a nested Python dictionary, modify, and then convert back:

1. `d = veh.to_pydict(flatten=False)` - convert to nested dict
2. Edit `d`
3. `veh = fsim.Vehicle.from_pydict(d)` - convert back to a FASTSim 3 Vehicle

In [ ]:
# Example modifying fuel converter peak power (+15%) and drag coefficient

# convert vehicle to nested dictionary
d = veh.to_pydict(flatten=False)

# Modify the nested dictionary
d["pt_type"]["Conv"]["fc"]["pwr_out_max_watts"] *= 1.15
d["chassis"]["drag_coef"] = 0.30

# Convert back to a Vehicle object
veh_modified = fsim.Vehicle.from_pydict(d)

## Loading a Drive Cycle

In FASTSim 2, cycles in FASTSim resources and custom cycles were loaded using `from_file`:

```python
resource_cyc = fsim.cycle.Cycle.from_file("udds") # loading a cycle from resource
custom_cyc = fsim.cycle.Cycle.from_file("path/to/custom_cycle.csv") # loading a custom cycle from a file
```

FASTSim 3 uses separate `from_resource` / `from_file` methods to load cycles, similar to how vehicles are loaded. You can use `list_resources` to view the cycles available in FASTSim resources:

In [ ]:
# List cycles in resources
print("Cycles provided in FASTSim resources:", fsim.Cycle.list_resources())

# loading a cycle from resources
cyc = fsim.Cycle.from_resource("udds.csv")

Cycles can also be loaded from file:
```python
cyc = fsim.Cycle.from_file("custom_cycle.csv")
```

## Running a Simulation

Four things changed:

1. **Class name:** `fsim.SimDrive` (was `fsim.simdrive.SimDrive`)
2. **Argument order:** `SimDrive(veh, cyc)` (was `SimDrive(cyc, veh)` -- order of inputs switched)
3. **Run method:** `sd.walk()` (was `sd.sim_drive()`)
4. **Save intervals:** configurable using `set_save_interval` (not configurable in FASTSim 2)

In [ ]:
# Create simdrive object and run a simulation
sd = fsim.SimDrive(veh, cyc)
sd.walk()

FASTSim 3 lets you configure how much time-series data is recorded via `set_save_interval` (see below). Saving less time-series data (or setting to None) allows for faster simulation when detailed time-series results aren't needed.

FASTSim 2 always recorded every time step. FASTSim 3 lets you trade time-series detail for speed:

- `veh.set_save_interval(1)` - record every step (default; needed for time-series plots).
- `veh.set_save_interval(n)` - record every *n*-th step.
- `veh.set_save_interval(None)` - disable per-step recording entirely. About 10× faster than FASTSim 2. Cumulative totals from `sd.to_pydict(flatten=True)` are still available.

Use `None` for parameter sweeps, large batch runs, and other applications where you only need aggregate results.

In [ ]:
# Fast run: no per-step recording, cumulative totals only
veh_fast = fsim.Vehicle.from_resource("2012_Ford_Fusion.yaml")
veh_fast.set_save_interval(None)

sd_fast = fsim.SimDrive(veh_fast, cyc)
sd_fast.walk()

# to_pydict still works - history arrays will be empty but state totals are populated
sd_dict = sd_fast.to_pydict(flatten=True)
print("Fuel energy:", sd_dict["veh.pt_type.Conv.fc.state.energy_fuel_joules"] / 3.6e6, "kWh")

## Reading Simulation Results

FASTSim 2 exposed results as array attributes on the `SimDrive` object:

```python
# FASTSim 2
sd.mph_ach              # numpy array of achieved speeds (mph)
sd.fc_kw_in_ach         # numpy array of fuel power in (kW)
sd.fc_kw_out_ach        # numpy array of fuel power out (kW)
sd.fs_kwh_out_ach[-1]   # total fuel energy consumed (kWh)
sd.soc                  # state of charge (for HEV/PHEV/BEV)
```

FASTSim 3 exposes results through two methods on `SimDrive`:

- **`sd.to_dataframe()`** - returns a Polars DataFrame (pass `pandas=True` for pandas). Recommended for accessing time series data. Column names are dot-separated paths mirroring the vehicle hierarchy.
- **`sd.to_pydict(flatten=True)`** - returns a flat `dict`. Used to access cumulative end-of-simulation totals which live under `*.state.*`. Available even when `set_save_interval(None)` is used.

In [ ]:
# Time-series results as a pandas DataFrame
df = sd.to_dataframe(pandas=True)

# FASTSim 2: sd.mph_ach
# FASTSim 3 (in m/s):
speed_ms = df["veh.history.speed_ach_meters_per_second"]

# FASTSim 2: sd.fc_kw_in_ach
# FASTSim 3 (in W):
fc_pwr_fuel_w = df["veh.pt_type.Conv.fc.history.pwr_fuel_watts"]

# FASTSim 2: sd.fc_kw_out_ach
# FASTSim 3 (in W):
fc_pwr_prop_w = df["veh.pt_type.Conv.fc.history.pwr_prop_watts"]

print(
    df[
        [
            "cyc.time_seconds",
            "veh.history.speed_ach_meters_per_second",
            "veh.pt_type.Conv.fc.history.pwr_fuel_watts",
            "veh.pt_type.Conv.fc.history.pwr_prop_watts",
        ]
    ].head(10)
)

In [ ]:
# Scalar cumulative totals via to_pydict(flatten=True).
# These are the FASTSim 3 equivalents of cumulative scalars accessed as the
# last element of an array in FASTSim 2 (e.g. sd.fs_kwh_out_ach[-1]).
sd_dict = sd.to_pydict(flatten=True)

fuel_energy_kwh = sd_dict["veh.pt_type.Conv.fc.state.energy_fuel_joules"] / 3.6e6
distance_km = sd_dict["veh.state.dist_meters"] / 1e3
cyc_met = sd_dict["veh.state.cyc_met_overall"]

print(f"Fuel energy consumed : {fuel_energy_kwh:.2f} kWh")
print(f"Distance driven      : {distance_km:.2f} km")
print(f"Fuel economy         : {distance_km / fuel_energy_kwh:.2f} km/kWh")
print(f"Cycle met throughout : {cyc_met}")

### Updating Result Paths

Common FASTSim 2 result attributes and their FASTSim 3 equivalents:

| FASTSim 2 | FASTSim 3 |
| --- | --- |
| `sd.cyc.mps` (target speed) | `sd.to_dataframe(pandas=True)["cyc.speed_meters_per_second"]` |
| `sd.mph_ach` | `sd.to_dataframe(pandas=True)["veh.history.speed_ach_meters_per_second"]` |
| `sd.dist_mi[-1]` | `sd.to_pydict(flatten=True)["veh.state.dist_meters"]` (in meters) |
| `sd.fc_kw_out_ach` | `sd.to_dataframe(pandas=True)["veh.pt_type.<Conv\|HEV\|PHEV>.fc.history.pwr_prop_watts"]` |
| `sd.fc_kw_in_ach` | `sd.to_dataframe(pandas=True)["veh.pt_type.<Conv\|HEV\|PHEV>.fc.history.pwr_fuel_watts"]` |
| `sd.fs_kwh_out_ach[-1]` | `sd.to_pydict(flatten=True)["veh.pt_type.<Conv\|HEV\|PHEV>.fc.state.energy_fuel_joules"]` |
| `sd.soc` | `sd.to_dataframe(pandas=True)["veh.pt_type.<HEV\|PHEV\|BEV>.res.history.soc"]` |
| `sd.ess_kw_out_ach` | `sd.to_dataframe(pandas=True)["veh.pt_type.<HEV\|PHEV\|BEV>.res.history.pwr_out_electrical_watts"]` |
| `sd.mc_kw_out_ach` | `sd.to_dataframe(pandas=True)["veh.pt_type.<HEV\|PHEV\|BEV>.em.history.pwr_prop_watts"]` |

Replace `<Conv|HEV|PHEV|BEV>` with the actual powertrain variant returned by `veh.veh_type()`.


(field-mapping-reference-guide)=
## Field-Mapping Reference Guide

The tables below map every FASTSim 2 `Vehicle` field to its FASTSim 3 equivalent. FASTSim 3 paths are the keys returned by `veh.to_pydict(flatten=True)`. Paths with `<variant>` change based on the powertrain type: `Conv`, `HEV`, `PHEV`, or `BEV`.

### Vehicle top-level attributes

*Present in all powertrain types.*

| FASTSim 2 field | Description | FASTSim 3 path | Notes |
| --- | --- | --- | --- |
| `scenario_name` | Vehicle name | `name` | |
| `veh_year` | Model year | `year` | |
| `veh_pt_type` | Powertrain type string | `pt_type` | Use `veh.veh_type()` to read it as a string. |
| `doc` | Free-form doc string | `doc` | |
| `selection` | Vehicle database ID | N/A | Not in FASTSim 3. |
| `veh_kg` | Total vehicle mass | `mass_kilograms` | In FASTSim 3, mass can be set at the top level (`mass_kilograms`) **or** derived automatically by summing all component masses (chassis + powertrain parts). |
| `veh_override_kg` | Override for total mass | N/A | Not in FASTSim 3. |
| `comp_mass_multiplier` | Multiplier used by FASTSim 2 mass calc | N/A | Not in FASTSim 3. |

### Chassis

*Present in all powertrain types. Fields live under `chassis.*`.*

| FASTSim 2 field | Description | FASTSim 3 path | Notes |
| --- | --- | --- | --- |
| `drag_coef` | Aerodynamic drag coefficient | `chassis.drag_coef` | |
| `frontal_area_m2` | Frontal area | `chassis.frontal_area_square_meters` | |
| `glider_kg` | Glider mass | `chassis.glider_mass_kilograms` | |
| `cargo_kg` | Cargo + passenger mass | `chassis.cargo_mass_kilograms` | |
| `veh_cg_m` | CG height (sign encodes drive type in F2) | `chassis.cg_height_meters` + `chassis.drive_type` | FASTSim 3 stores `abs(veh_cg_m)` as `cg_height_meters`; sign decoded into `drive_type` (`FWD`/`RWD`/`AWD`). |
| `drive_axle_weight_frac` | Weight fraction on drive axle | `chassis.drive_axle_weight_frac` | |
| `wheel_base_m` | Wheelbase | `chassis.wheel_base_meters` | |
| `wheel_inertia_kg_m2` | Per-wheel rotational inertia | `chassis.wheel_inertia_kilogram_square_meters` | |
| `num_wheels` | Number of wheels | `chassis.num_wheels` | |
| `wheel_rr_coef` | Rolling-resistance coefficient | `chassis.wheel_rr_coef` | |
| `wheel_radius_m` | Wheel radius | `chassis.wheel_radius_meters` | |
| `wheel_coef_of_fric` | Wheel–road friction coefficient | `chassis.wheel_fric_coef` | |
| N/A | Tire designation | `chassis.tire_code` | New in FASTSim 3. |

### Fuel storage

*Present in: `Conv`, `HEV`, `PHEV`. Not present in: `BEV`.*

| FASTSim 2 field | Description | FASTSim 3 path | Notes |
| --- | --- | --- | --- |
| `fs_max_kw` | FS peak output power | `pt_type.<variant>.fs.pwr_out_max_watts` | kW → W. |
| `fs_secs_to_peak_pwr` | FS ramp-up time | `pt_type.<variant>.fs.pwr_ramp_lag_seconds` | |
| `fs_kwh` | FS energy capacity | `pt_type.<variant>.fs.energy_capacity_joules` | kWh → J. |
| `fs_kwh_per_kg` | Fuel specific energy | `pt_type.<variant>.fs.specific_energy_joules_per_kilogram` ||
| `fs_mass_kg` | Derived FS mass | `pt_type.<variant>.fs.mass_kilograms` ||

### Fuel converter

*Present in: `Conv`, `HEV`, `PHEV`. Not present in: `BEV`.*

| FASTSim 2 field | Description | FASTSim 3 path | Notes |
| --- | --- | --- | --- |
| `fc_max_kw` | FC peak continuous power | `pt_type.<variant>.fc.pwr_out_max_watts` | kW → W. |
| `fc_sec_to_peak_pwr` | FC ramp-up time | `pt_type.<variant>.fc.pwr_ramp_lag_seconds` | |
| `fc_eff_map` | FC efficiency map (y values) | `pt_type.<variant>.fc.eff_interp_from_pwr_out` (values) | |
| `fc_pwr_out_perc` | FC output-power fraction x-grid | `pt_type.<variant>.fc.eff_interp_from_pwr_out` (grid) | |
| `fc_eff_type` | `SI`/`Atkinson`/`Diesel`/`H2FC`/`HD_Diesel` | N/A | Not in FASTSim 3. |
| `fc_base_kg`, `fc_kw_per_kg` | FC mass model | N/A | Not in FASTSim 3.  |
| `fc_mass_kg` | Derived FC mass | `pt_type.<variant>.fc.mass_kilograms` | |
| `idle_fc_kw` | FC idle fuel power | `pt_type.<variant>.fc.pwr_idle_fuel_watts` | |
| `min_fc_time_on` | Min FC on-time before shutoff | `pt_type.<variant>.pt_cntrl.RGWDB.fc_min_time_on_seconds` | Lives in the powertrain controller, not on the FC itself. |
| `fc_peak_eff_override` | Curve-scaling override | N/A | Not in FASTSim 3. |

### Reversible energy storage (battery)

*Present in: `HEV`, `PHEV`, `BEV`. Not present in: `Conv`.*

| FASTSim 2 field | Description | FASTSim 3 path | Notes |
| --- | --- | --- | --- |
| `ess_max_kw` | ESS peak power | `pt_type.<variant>.res.pwr_out_max_watts` | kW → W. |
| `ess_max_kwh` | ESS energy capacity | `pt_type.<variant>.res.energy_capacity_joules` | kWh → J. |
| `ess_round_trip_eff` | Round-trip efficiency | `pt_type.<variant>.res.eff_interp` | Stored as a constant one-way efficiency `sqrt(ess_round_trip_eff)`. |
| `min_soc` / `max_soc` | SOC limits | `pt_type.<variant>.res.min_soc` / `.max_soc` | |
| `ess_kg_per_kwh`, `ess_base_kg` | ESS mass model | N/A | Not in FASTSim 3. |
| `ess_mass_kg` | Derived ESS mass | `pt_type.<variant>.res.mass_kilograms` | |
| `ess_life_coef_a`, `ess_life_coef_b` | Battery life fit | N/A | Not in FASTSim 3. |
| `ess_dischg_to_fc_max_eff_perc`, `ess_chg_to_fc_max_eff_perc`, `ess_to_fuel_ok_error` | Hybrid SOC-balancing knobs | N/A | Not in FASTSim 3. |
| `max_regen`, `max_regen_kwh`, `regen_a`, `regen_b` | Regen model | N/A | Not in FASTSim 3; FASTSim 3 handles regen limits via powertrain controls. |

### Electric machine (motor)

*Present in: `HEV`, `PHEV`, `BEV`. Not present in: `Conv`.*

| FASTSim 2 field | Description | FASTSim 3 path | Notes |
| --- | --- | --- | --- |
| `mc_max_kw` | Motor peak continuous power | `pt_type.<variant>.em.pwr_out_max_watts` | kW → W. |
| `mc_eff_map` | efficiency array | `pt_type.<variant>.em.eff_interp_achieved` (values) | |
| `mc_pwr_out_perc` | Motor output-fraction x-grid | `pt_type.<variant>.em.eff_interp_achieved` (grid) | |
| `mc_sec_to_peak_pwr` | Motor ramp-up time | N/A | Not in FASTSim 3. |
| `mc_mass_kg` | Derived motor mass | `pt_type.<variant>.em.mass_kilograms` | |
| `mc_pe_base_kg`, `mc_pe_kg_per_kw` | Power-electronics mass model | N/A | Not in FASTSim 3. |
| `mc_peak_eff_override` | Curve-scaling override | N/A | Not in FASTSim 3. |

### Transmission, aux loads, HEV controls

**Transmission** (all powertrain types):

| FASTSim 2 field | FASTSim 3 path | Notes |
| --- | --- | --- |
| `trans_eff` | `pt_type.<variant>.transmission.eff_interp` | Stored as a constant efficiency. |
| `trans_kg` | `pt_type.<variant>.transmission.mass_kilograms` | |

**Auxiliary loads:**

| FASTSim 2 field | FASTSim 3 path | Notes |
| --- | --- | --- |
| `aux_kw` | `pwr_aux_base_watts` | kW → W. |
| `alt_eff` | `pt_type.Conv.alt_eff` | Only on `Conv`; implicitly `1.0` on others. |
| `chg_eff` | N/A | Not in FASTSim 3. |

**HEV / PHEV powertrain controls** (present on `HEV`, `PHEV`):

| FASTSim 2 field | FASTSim 3 path | Notes |
| --- | --- | --- |
| `mph_fc_on` | `pt_type.<variant>.pt_cntrl.RGWDB.speed_fc_forced_on_meters_per_second` | mph → m/s. |
| `kw_demand_fc_on` | `pt_type.<variant>.pt_cntrl.RGWDB.frac_pwr_demand_fc_forced_on` | kW → fraction. |
| `min_fc_time_on` | `pt_type.<variant>.pt_cntrl.RGWDB.fc_min_time_on_seconds` | |
| `stop_start` | `pt_type.<variant>.pt_cntrl.StopStart.*` | Separate powertrain controller type in FASTSim 3 (for both `Conv` and `HEV`). Activate with `veh.use_stop_start_controller()`. |
| `force_aux_on_fc` | `pt_type.<variant>.aux_cntrl` | `AuxOnFcPriority` (FC handles aux) or `AuxOnResPriority` (battery handles aux if feasible, default). |
| `max_accel_buffer_mph`, `max_accel_buffer_perc_of_useable_soc`, `perc_high_acc_buf` | See below |  |

The FASTSim 2 acceleration SOC-buffer fields are replaced by six RGWDB tuning fields (all under `pt_type.<variant>.pt_cntrl.RGWDB.*`). Each buffer is defined by a reference speed (at which the buffer reaches its full size) and a coefficient that scales the buffer magnitude:

| FASTSim 3 RGWDB field | Description |
| --- | --- |
| `speed_soc_disch_buffer_meters_per_second` | Reference speed for discharge / acceleration buffer |
| `speed_soc_disch_buffer_coeff` | Coefficient scaling the discharge buffer |
| `speed_soc_fc_on_buffer_meters_per_second` | Reference speed for the SOC threshold that forces the FC on |
| `speed_soc_fc_on_buffer_coeff` | Coefficient scaling the FC-on buffer |
| `speed_soc_regen_buffer_meters_per_second` | Reference speed for regen / charging buffer |
| `speed_soc_regen_buffer_coeff` | Coefficient scaling the regen buffer |

**Validation reference values** (`val_udds_mpgge`, `val_hwy_mpgge`, `val_comb_mpgge`, `val0_to60_mph`, `val_range_miles`, …) are **not** carried over to FASTSim 3.